### Step 0. 기본 설정

In [2]:
from pathlib import Path
import re
import math
import json
import csv
import pandas as pd

BASE_DIR = Path(r"D:\PyProject\AIFFEL_AI\LLM\Main_Quest\ACE\data_01")
DXF_FILE = BASE_DIR / "도면1.dxf"
OUTPUT_DIR = BASE_DIR / "output_hbeam"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(DXF_FILE.exists(), DXF_FILE)

True D:\PyProject\AIFFEL_AI\LLM\Main_Quest\ACE\data_01\도면1.dxf


### STEP 1. DXF 원문 읽기

In [5]:
def read_dxf_lines(dxf_path: Path):
    if not dxf_path.exists():
        raise FileNotFoundError(f"DXF 파일을 찾을 수 없습니다: {dxf_path}")
    return dxf_path.read_text(errors="ignore").splitlines()

lines = read_dxf_lines(DXF_FILE)
print("DXF line 수:", len(lines))

DXF line 수: 1548092


### STEP 2. TEXT / MTEXT 추출

In [8]:
def normalize_dxf_text(text: str) -> str:
    text = text.replace("\\P", " ").replace("\\X", " ").replace("\\~", " ")
    text = re.sub(r"\\[A-Za-z0-9]+;?", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def extract_text_entities(lines):
    records = []
    i = 0

    while i < len(lines) - 1:
        code = lines[i].strip()
        value = lines[i + 1].rstrip("\n")

        if code == "0" and value in ("TEXT", "MTEXT"):
            entity_type = value
            j = i + 2

            entity = {
                "entity_type": entity_type,
                "layer": "",
                "text": "",
                "x": None,
                "y": None,
            }

            while j < len(lines) - 1:
                group_code = lines[j].strip()
                group_value = lines[j + 1].rstrip("\n")

                if group_code == "0":
                    break

                if group_code == "8":
                    entity["layer"] = group_value

                elif group_code == "10":
                    try:
                        entity["x"] = float(group_value)
                    except:
                        pass

                elif group_code == "20":
                    try:
                        entity["y"] = float(group_value)
                    except:
                        pass

                elif entity_type == "TEXT" and group_code == "1":
                    entity["text"] = group_value

                elif entity_type == "MTEXT" and group_code in ("1", "3"):
                    entity["text"] += group_value

                j += 2

            if entity["text"].strip():
                entity["normalized"] = normalize_dxf_text(entity["text"])
                records.append(entity)

            i = j
        else:
            i += 2

    return records


text_records = extract_text_entities(lines)
print("TEXT/MTEXT 개수:", len(text_records))

pd.DataFrame(text_records).head()

TEXT/MTEXT 개수: 4315


,entity_type,layer,text,x,y,normalized
0,TEXT,HATCH,도면번호,1.459348e+06,-279949.875557,도면번호
1,TEXT,HATCH,SHEET NO,1.459348e+06,-279683.192283,SHEET NO
2,TEXT,PAPER,승 인,1.448968e+06,-279329.155040,승 인
3,TEXT,PAPER,APPROVED BY,1.448953e+06,-279034.379337,APPROVED BY
4,TEXT,PAPER,PROJECT TITLE,1.452809e+06,-279033.822312,PROJECT TITLE


### STEP 3. H빔 규격만 추출

In [11]:
def canonicalize_h_beam(text: str):
    s = text.upper().replace(" ", "")
    s = s.replace("×", "X")

    m = re.search(
        r"H-?(\d+(?:\.\d+)?)X(\d+(?:\.\d+)?)X(\d+(?:\.\d+)?)(?:/(\d+(?:\.\d+)?))?",
        s,
    )

    if not m:
        return None

    h, b, tw, tf = m.groups()

    if tf:
        return f"H-{h}x{b}x{tw}/{tf}"
    else:
        return f"H-{h}x{b}x{tw}"


h_texts = []

for row in text_records:
    spec = canonicalize_h_beam(row["normalized"])
    if spec:
        new_row = row.copy()
        new_row["h_beam_spec"] = spec
        h_texts.append(new_row)

df_h_texts = pd.DataFrame(h_texts)

print("H빔 텍스트 개수:", len(df_h_texts))
display(df_h_texts[["layer", "normalized", "h_beam_spec", "x", "y"]].head(20))

H빔 텍스트 개수: 44


,layer,normalized,h_beam_spec,x,y
0,TXT,H- 588x300x12/20,H-588x300x12/20,169206.628840,1.077651e+06
1,TXT,H- 200x200x8/12,H-200x200x8/12,169218.835422,1.077138e+06
2,TXT,H- 250x125x6/9,H-250x125x6/9,292010.723179,1.077651e+06
3,TXT,H- 600x200x11/17,H-600x200x11/17,292002.357789,1.078136e+06
4,TXT,H- 600x200x11/17,H-600x200x11/17,292002.357789,1.079590e+06
5,TXT,H- 600x200x11/17,H-600x200x11/17,292002.357789,1.080075e+06
6,TXT,H- 300x150x6.5/9,H-300x150x6.5/9,292027.842600,1.079105e+06
7,TXT,H- 200x100x5.5/8,H-200x100x5.5/8,292027.842600,1.077166e+06
8,TXT,H- 250x125x6/9,H-250x125x6/9,373867.711850,1.077651e+06
9,TXT,H- 600x200x11/17,H-600x200x11/17,373862.874566,1.078136e+06


### STEP 4. H빔 규격별 텍스트 출현 수 확인

In [14]:
df_h_count = (
    df_h_texts
    .groupby("h_beam_spec")
    .size()
    .reset_index(name="text_count")
    .sort_values("text_count", ascending=False)
)

display(df_h_count)

,h_beam_spec,text_count
10,H-600x200x11/17,9
4,H-300x150x6.5/9,6
1,H-200x200x8/12,5
2,H-250x125x6/9,5
7,H-400x200x8/13,4
9,H-588x300x12/20,4
0,H-200x100x5.5/8,3
3,H-250x250x9/14,3
8,H-440x300x11/18,3
6,H-400x200x8,1


### STEP 5. LINE 객체 추출

In [17]:
def extract_line_entities(lines):
    records = []
    i = 0

    while i < len(lines) - 1:
        code = lines[i].strip()
        value = lines[i + 1].rstrip("\n")

        if code == "0" and value == "LINE":
            j = i + 2

            entity = {
                "entity_type": "LINE",
                "layer": "",
                "x1": None,
                "y1": None,
                "x2": None,
                "y2": None,
            }

            while j < len(lines) - 1:
                group_code = lines[j].strip()
                group_value = lines[j + 1].rstrip("\n")

                if group_code == "0":
                    break

                try:
                    if group_code == "8":
                        entity["layer"] = group_value
                    elif group_code == "10":
                        entity["x1"] = float(group_value)
                    elif group_code == "20":
                        entity["y1"] = float(group_value)
                    elif group_code == "11":
                        entity["x2"] = float(group_value)
                    elif group_code == "21":
                        entity["y2"] = float(group_value)
                except:
                    pass

                j += 2

            if None not in [entity["x1"], entity["y1"], entity["x2"], entity["y2"]]:
                dx = entity["x2"] - entity["x1"]
                dy = entity["y2"] - entity["y1"]
                entity["length_dxf"] = math.sqrt(dx**2 + dy**2)
                entity["cx"] = (entity["x1"] + entity["x2"]) / 2
                entity["cy"] = (entity["y1"] + entity["y2"]) / 2
                records.append(entity)

            i = j
        else:
            i += 2

    return records


line_records = extract_line_entities(lines)
df_lines = pd.DataFrame(line_records)

print("LINE 개수:", len(df_lines))
display(df_lines.head())

LINE 개수: 15339


,entity_type,layer,x1,y1,x2,y2,length_dxf,cx,cy
0,LINE,0,9.0,-261.0,9.0,-54.0,207.0,9.0,-157.5
1,LINE,0,441.0,-54.0,441.0,-261.0,207.0,441.0,-157.5
2,LINE,0,225.0,-144.0,225.0,-54.0,90.0,225.0,-99.0
3,LINE,0,225.0,-180.0,225.0,-162.0,18.0,225.0,-171.0
4,LINE,0,36.0,-27.0,414.0,-27.0,378.0,225.0,-27.0


### STEP 6. H빔 텍스트와 가장 가까운 LINE 연결

In [20]:
def distance_point_to_line_center(px, py, line):
    return math.sqrt((px - line["cx"])**2 + (py - line["cy"])**2)


def match_h_text_to_nearest_line(df_h_texts, df_lines, max_distance=3000):
    matched = []

    line_dicts = df_lines.to_dict("records")

    for _, hrow in df_h_texts.iterrows():
        px, py = hrow["x"], hrow["y"]

        if pd.isna(px) or pd.isna(py):
            continue

        best_line = None
        best_dist = float("inf")

        for line in line_dicts:
            dist = distance_point_to_line_center(px, py, line)
            if dist < best_dist:
                best_dist = dist
                best_line = line

        if best_line and best_dist <= max_distance:
            matched.append({
                "h_beam_spec": hrow["h_beam_spec"],
                "text": hrow["normalized"],
                "text_layer": hrow["layer"],
                "text_x": px,
                "text_y": py,
                "line_layer": best_line["layer"],
                "line_x1": best_line["x1"],
                "line_y1": best_line["y1"],
                "line_x2": best_line["x2"],
                "line_y2": best_line["y2"],
                "line_length_dxf": best_line["length_dxf"],
                "match_distance": best_dist,
            })

    return pd.DataFrame(matched)


df_matched = match_h_text_to_nearest_line(df_h_texts, df_lines, max_distance=3000)

print("매칭된 H빔-선 개수:", len(df_matched))
display(df_matched.head(20))

매칭된 H빔-선 개수: 44


,h_beam_spec,text,text_layer,text_x,text_y,line_layer,line_x1,line_y1,line_x2,line_y2,line_length_dxf,match_distance
0,H-588x300x12/20,H- 588x300x12/20,TXT,169206.628840,1.077651e+06,PAPER,168997.537139,1.078481e+06,168997.537139,1.076980e+06,1500.813227,223.673209
1,H-200x200x8/12,H- 200x200x8/12,TXT,169218.835422,1.077138e+06,PAPER,168997.537139,1.078481e+06,168997.537139,1.076980e+06,1500.813227,632.754875
2,H-250x125x6/9,H- 250x125x6/9,TXT,292010.723179,1.077651e+06,PAPER,289136.783720,1.076980e+06,296277.036235,1.076980e+06,7140.252516,966.889446
3,H-600x200x11/17,H- 600x200x11/17,TXT,292002.357789,1.078136e+06,PAPER,291793.266087,1.080905e+06,291793.266087,1.076980e+06,3925.012315,833.354814
4,H-600x200x11/17,H- 600x200x11/17,TXT,292002.357789,1.079590e+06,PAPER,291793.266087,1.080905e+06,291793.266087,1.076980e+06,3925.012315,680.729449
5,H-600x200x11/17,H- 600x200x11/17,TXT,292002.357789,1.080075e+06,PAPER,289136.783720,1.080374e+06,296277.036235,1.080374e+06,7140.252516,765.259397
6,H-300x150x6.5/9,H- 300x150x6.5/9,TXT,292027.842600,1.079105e+06,PAPER,291793.266087,1.080905e+06,291793.266087,1.076980e+06,3925.012315,285.638420
7,H-200x100x5.5/8,H- 200x100x5.5/8,TXT,292027.842600,1.077166e+06,PAPER,289136.783720,1.076980e+06,296277.036235,1.076980e+06,7140.252516,704.113986
8,H-250x125x6/9,H- 250x125x6/9,TXT,373867.711850,1.077651e+06,PAPER,370997.300497,1.076980e+06,378137.553013,1.076980e+06,7140.252516,969.432871
9,H-600x200x11/17,H- 600x200x11/17,TXT,373862.874566,1.078136e+06,PAPER,373653.782864,1.080905e+06,373653.782864,1.076980e+06,3925.012315,833.354814


### STEP 7. DXF 단위를 m로 변환

In [23]:
DXF_UNIT_PER_METER = 1000

df_matched["length_m"] = df_matched["line_length_dxf"] / DXF_UNIT_PER_METER

display(df_matched[[
    "h_beam_spec",
    "length_m",
    "match_distance",
    "text",
    "line_layer"
]].head(20))

,h_beam_spec,length_m,match_distance,text,line_layer
0,H-588x300x12/20,1.500813,223.673209,H- 588x300x12/20,PAPER
1,H-200x200x8/12,1.500813,632.754875,H- 200x200x8/12,PAPER
2,H-250x125x6/9,7.140253,966.889446,H- 250x125x6/9,PAPER
3,H-600x200x11/17,3.925012,833.354814,H- 600x200x11/17,PAPER
4,H-600x200x11/17,3.925012,680.729449,H- 600x200x11/17,PAPER
5,H-600x200x11/17,7.140253,765.259397,H- 600x200x11/17,PAPER
6,H-300x150x6.5/9,3.925012,285.638420,H- 300x150x6.5/9,PAPER
7,H-200x100x5.5/8,7.140253,704.113986,H- 200x100x5.5/8,PAPER
8,H-250x125x6/9,7.140253,969.432871,H- 250x125x6/9,PAPER
9,H-600x200x11/17,3.925012,833.354814,H- 600x200x11/17,PAPER


### STEP 8. H형강 단위중량 계산

In [26]:
def parse_h_beam_spec(spec: str):
    """
    H-600x200x11/17
    return h, b, tw, tf
    """
    s = spec.upper().replace(" ", "")
    m = re.search(
        r"H-(\d+(?:\.\d+)?)X(\d+(?:\.\d+)?)X(\d+(?:\.\d+)?)/(\d+(?:\.\d+)?)",
        s
    )

    if not m:
        return None

    h, b, tw, tf = map(float, m.groups())
    return h, b, tw, tf


def estimate_h_beam_kg_per_m(spec: str):
    """
    단순 근사:
    단면적(mm2) = 2 * B * tf + (H - 2tf) * tw
    kg/m = 단면적(mm2) * 0.00785
    """
    parsed = parse_h_beam_spec(spec)

    if not parsed:
        return None

    h, b, tw, tf = parsed

    area_mm2 = 2 * b * tf + (h - 2 * tf) * tw
    kg_per_m = area_mm2 * 0.00785

    return kg_per_m


df_matched["kg_per_m_est"] = df_matched["h_beam_spec"].apply(estimate_h_beam_kg_per_m)
df_matched["weight_kg_est"] = df_matched["length_m"] * df_matched["kg_per_m_est"]
df_matched["weight_ton_est"] = df_matched["weight_kg_est"] / 1000

display(df_matched[[
    "h_beam_spec",
    "length_m",
    "kg_per_m_est",
    "weight_kg_est",
    "weight_ton_est",
    "match_distance"
]].head(20))

,h_beam_spec,length_m,kg_per_m_est,weight_kg_est,weight_ton_est,match_distance
0,H-588x300x12/20,1.500813,145.82160,218.850986,0.218851,223.673209
1,H-200x200x8/12,1.500813,48.73280,73.138831,0.073139,632.754875
2,H-250x125x6/9,7.140253,28.58970,204.137677,0.204138,966.889446
3,H-600x200x11/17,3.925012,102.25410,401.348602,0.401349,833.354814
4,H-600x200x11/17,3.925012,102.25410,401.348602,0.401349,680.729449
5,H-600x200x11/17,7.140253,102.25410,730.120095,0.730120,765.259397
6,H-300x150x6.5/9,3.925012,35.58405,139.667834,0.139668,285.638420
7,H-200x100x5.5/8,7.140253,20.50420,146.405166,0.146405,704.113986
8,H-250x125x6/9,7.140253,28.58970,204.137677,0.204138,969.432871
9,H-600x200x11/17,3.925012,102.25410,401.348602,0.401349,833.354814


### STEP 9. 규격별 H빔 물량 집계

In [29]:
df_hbeam_quantity = (
    df_matched
    .groupby("h_beam_spec")
    .agg(
        matched_count=("h_beam_spec", "count"),
        total_length_m=("length_m", "sum"),
        avg_length_m=("length_m", "mean"),
        kg_per_m_est=("kg_per_m_est", "mean"),
        total_weight_kg_est=("weight_kg_est", "sum"),
        total_weight_ton_est=("weight_ton_est", "sum"),
    )
    .reset_index()
    .sort_values("total_weight_ton_est", ascending=False)
)

display(df_hbeam_quantity)

,h_beam_spec,matched_count,total_length_m,avg_length_m,kg_per_m_est,total_weight_kg_est,total_weight_ton_est
10,H-600x200x11/17,9,44.970831,4.996759,102.25410,4598.451895,4.598452
4,H-300x150x6.5/9,6,31.040183,5.173364,35.58405,1104.535439,1.104535
9,H-588x300x12/20,4,6.003253,1.500813,145.82160,875.403944,0.875404
8,H-440x300x11/18,3,6.670331,2.223444,119.66540,798.207795,0.798208
2,H-250x125x6/9,5,24.994933,4.998987,28.58970,714.597637,0.714598
7,H-400x200x8/13,4,8.457418,2.114355,64.30720,543.872901,0.543873
3,H-250x250x9/14,3,6.670331,2.223444,70.63430,471.154142,0.471154
0,H-200x100x5.5/8,3,21.420758,7.140253,20.50420,439.215497,0.439215
1,H-200x200x8/12,5,7.353985,1.470797,48.73280,358.380271,0.358380
5,H-350x350x12/19,1,1.350732,1.350732,133.79540,180.721715,0.180722


### STEP 10. 전체 H빔 물량 합계

In [32]:
total_length = df_hbeam_quantity["total_length_m"].sum()
total_kg = df_hbeam_quantity["total_weight_kg_est"].sum()
total_ton = df_hbeam_quantity["total_weight_ton_est"].sum()

summary = {
    "dxf_file": str(DXF_FILE),
    "h_beam_text_count": len(df_h_texts),
    "matched_h_beam_count": len(df_matched),
    "total_length_m_est": total_length,
    "total_weight_kg_est": total_kg,
    "total_weight_ton_est": total_ton,
}

print(json.dumps(summary, ensure_ascii=False, indent=2))

{
  "dxf_file": "D:\\PyProject\\AIFFEL_AI\\LLM\\Main_Quest\\ACE\\data_01\\도면1.dxf",
  "h_beam_text_count": 44,
  "matched_h_beam_count": 44,
  "total_length_m_est": 165.3552522701154,
  "total_weight_kg_est": 10084.541234742597,
  "total_weight_ton_est": 10.084541234742598
}


### 도면1.dxf 분석 결과 (H빔 기준)
---
- H빔 규격 텍스트 검출: 44개
- H빔과 연결된 선(Line) 매칭: 44개 성공
- H빔 총 길이 추정: 약 165.36m
- H빔 총 중량 추정: 약 10,084kg
- H빔 총 물량 추정: 약 10.08톤

### STEP 11. 결과 저장

In [ ]:
# df_h_texts.to_csv(OUTPUT_DIR / "step_01_h_beam_texts.csv", index=False, encoding="utf-8-sig")
# df_matched.to_csv(OUTPUT_DIR / "step_02_h_beam_line_matched.csv", index=False, encoding="utf-8-sig")
# df_hbeam_quantity.to_csv(OUTPUT_DIR / "step_03_h_beam_quantity_summary.csv", index=False, encoding="utf-8-sig")

# with open(OUTPUT_DIR / "step_04_h_beam_summary.json", "w", encoding="utf-8") as f:
#     json.dump(summary, f, ensure_ascii=False, indent=2)

# print("저장 완료:", OUTPUT_DIR)

도면1에서는 H빔(철골) 외에도 구조 물량과 연결될 가능성이 있는 요소들이 존재할 가능성이 매우 높습니다.

구조 적산 관점에서 보면 물량 요소는 크게 4개입니다.

구분	대표 물량 단위
철골	ton, kg
철근	ton, kg
콘크리트	m³
거푸집	m²

현재는 H빔만 찾았지만, 실제로 도면 안에는 아래 요소들도 숨어 있을 가능성이 큽니다.

1. 철골(H빔 외)

현재 발견:

H-600x200x11/17

같은 H형강.

추가로 탐색해야 할 철골 요소:

종류	예시
BH	BH-700x300x13/24
BOX	BOX-200x200x9
PIPE	PIPE-165.2x6
C형강	C-100x50x20x2.3
각파이프	□-150x150x6
앵글	L-75x75x6

즉 현재는 H빔만 봤고,
철골 전체는 아닙니다.

2. 철근 물량 요소

도면에서 가장 중요한 물량 중 하나입니다.

보통 이런 형태로 존재합니다.

예시	의미
D13@200	13mm 철근 200간격
HD16	고장력 철근
4-D22	D22 철근 4개
STIRRUP D10@150	스터럽 철근

이 정보로:

철근 길이 × 단위중량

계산해서 ton 산출합니다.

3. 콘크리트 물량 요소

콘크리트는 보통:

요소	의미
슬라브(SLAB)	면적 × 두께
보(BEAM)	단면 × 길이
기둥(COLUMN)	단면 × 높이
기초(FOOTING)	체적

으로 계산합니다.

도면 안에는 보통:

THK=200
SLAB 150

같은 정보가 존재합니다.

4. 거푸집 물량 요소

거푸집은:

콘크리트 접촉면적

입니다.

예:

부재	거푸집 계산
보	측면 + 하부
기둥	4면
슬라브	하부

즉:

부재 형상 + 길이

가 필요합니다.